# Fine-Tuned LLM for Product Price Prediction


This projec fine-tunes Llama 3.2 3B using QLoRA on the `ed-donner/items_prompts_full` dataset to predict product prices from text descriptions. A Weighted Top-K method was used to sample the model's next-token probability distribution, interpreting numeric tokens as prices, and returning a probability-weighted average. 

| Predictor | Error |
|---|---|
| Human Level | $87.62 |
| Ed's Fine-Tuned Llama 3.2 3B | $65.40 |
| ChatGPT | $62.51 |
| **This approach** | **$36.89** |

This approach outperforms the instructor baseline by ~$28, ChatGPT by ~$26, and human level by ~$51 — achieving a 76% hit rate, with the main weakness being underestimation of high-value products.



In [ ]:
# imports

import sys
import os
import re
import math
from dotenv import load_dotenv
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
from peft import PeftModel
sys.path.append(os.path.abspath("../../.."))
from week7.util import evaluate

import torch
import torch.nn.functional as F

from typing import Callable, List, Dict, Any

import math
import matplotlib.pyplot as plt

import pandas as pd
from tabulate import tabulate

In [ ]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
HF_USER = "ed-donner" # Huggingface username

LITE_MODE = False

DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

if LITE_MODE:
  RUN_NAME = "2025-11-30_15.10.55-lite"
  REVISION = None
else:
  RUN_NAME = "2025-11-28_18.47.07"
  REVISION = "b19c8bfea3b6ff62237fbb0a8da9779fc12cefbd"

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# Hyper-parameters - QLoRA

HAS_CUDA = torch.cuda.is_available()
DEVICE = "cuda" if HAS_CUDA else "cpu"

if HAS_CUDA:
  capability = torch.cuda.get_device_capability()
  use_bf16 = capability[0] >= 8
  QUANT_4_BIT = True
else:
  capability = (0, 0)
  use_bf16 = False
  QUANT_4_BIT = False
  print("CUDA not available - using CPU mode without 4-bit quantization.")

### Log in to HuggingFace

If you don't already have a HuggingFace account, visit https://huggingface.co to sign up and create a token.

Then select the Secrets for this Notebook by clicking on the key icon in the left, and add a new secret called `HF_TOKEN` with the value as your token.


In [ ]:
# Log in to HuggingFace
load_dotenv(override=True)
hf_token = os.getenv('HUGGING_FACE_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
dataset = load_dataset(DATASET_NAME)
test = dataset['test']

In [ ]:
test[0]

### Tokenizer and Model

In [ ]:
# pick the right quantization

if HAS_CUDA and QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
elif HAS_CUDA:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )
else:
  quant_config = None

In [ ]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

if quant_config is not None:
  base_model = AutoModelForCausalLM.from_pretrained(
      BASE_MODEL,
      quantization_config=quant_config,
      device_map="auto",
  )
else:
  base_model = AutoModelForCausalLM.from_pretrained(
      BASE_MODEL,
      torch_dtype=torch.float32,
  ).to(DEVICE)

base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Load the fine-tuned model with PEFT
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)

if quant_config is None:
  fine_tuned_model = fine_tuned_model.to(DEVICE)

print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
fine_tuned_model

### Ed's Model's Performance

In [ ]:
def model_predict(item):
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

In [ ]:
set_seed(42)
evaluate(model_predict, test)

- Ed's Error: $65.40
- ChatGPT: $62.51
- Human Level: $87.62

### Improving the Performance

In [ ]:
def extract_price_value(text):
    """Extract price as float from a string containing 'Price is $...'."""
    match = re.search(r'Price is \$"?([\d,]+(?:\.\d+)?)"?', text)
    if match:
        return float(match.group(1).replace(',', ''))
    return 0.0

In [ ]:
def generate_price_from_prompt(prompt):
    """Generate a response using the fine‑tuned model and parse the price."""
    set_seed(42)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    outputs = fine_tuned_model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,          # greedy decoding for speed and consistency
        num_beams=1
    )
    response = tokenizer.decode(outputs[0])
    return extract_price_value(response)

In [ ]:
def weighted_topk_price_prediction(prompt, device=DEVICE, top_k=5):
    """
    Compute weighted average of top‑K token probabilities interpreted as prices.
    Returns the weighted average, or 0.0 if no valid price tokens are found.
    """
    set_seed(42)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = fine_tuned_model(**inputs).logits[:, -1, :]
    probs = F.softmax(logits, dim=-1)
    top_probs, top_token_ids = probs.topk(top_k)

    # Gather valid prices and their probabilities
    valid = []
    for prob, token_id in zip(top_probs[0], top_token_ids[0]):
        token_str = tokenizer.decode(token_id).strip()
        try:
            price = float(token_str)
            if price > 0:
                valid.append((price, prob.item()))
        except ValueError:
            continue

    if not valid:
        return 0.0

    total_weight = sum(w for _, w in valid)
    weighted_sum = sum(p * w for p, w in valid)
    return weighted_sum / total_weight

### Performance Results

In [ ]:
class PricePredictorTester:
    """
    Evaluate a price prediction function against ground truth data.
    Stores results, computes errors, and generates a scatter plot.
    """
    def __init__(self, predictor: Callable, data: List[Dict[str, Any]], name: str = None, size: int = 250):
        self.predictor = predictor
        self.data = data[:size]
        self.name = name or predictor.__name__.replace('_', ' ').title()
        self.size = size
        self.guesses: List[float] = []
        self.truths: List[float] = []
        self.errors: List[float] = []
        self.sles: List[float] = []
        self.colors: List[str] = []

    def _get_color(self, error: float, truth: float) -> str:
        rel_error = error / truth if truth != 0 else float('inf')
        if error < 40 or rel_error < 0.2:
            return 'green'
        if error < 80 or rel_error < 0.4:
            return 'orange'
        return 'red'

    def run_test(self) -> None:
        for i, dp in enumerate(self.data, 1):
            # Use correct keys: 'prompt' and 'completion'
            guess = self.predictor(dp['prompt'])
            truth = float(dp['completion'])
            error = abs(guess - truth)
            sle = (math.log(truth + 1) - math.log(guess + 1)) ** 2
            color = self._get_color(error, truth)
            self.guesses.append(guess)
            self.truths.append(truth)
            self.errors.append(error)
            self.sles.append(sle)
            self.colors.append(color)
            print(f"{i:3}: Guess=${guess:7.2f} Truth=${truth:7.2f} Error=${error:6.2f} SLE={sle:5.2f}")
        self._plot_results()

    def _plot_results(self) -> None:
        avg_error = sum(self.errors) / self.size
        rmsle = math.sqrt(sum(self.sles) / self.size)
        hit_rate = sum(1 for c in self.colors if c == 'green') / self.size * 100
        plt.figure(figsize=(12, 8))
        max_val = max(max(self.truths), max(self.guesses))
        plt.plot([0, max_val], [0, max_val], 'deepskyblue', lw=2, alpha=0.6)
        plt.scatter(self.truths, self.guesses, c=self.colors, s=3)
        plt.xlabel('Ground Truth ($)')
        plt.ylabel('Model Estimate ($)')
        plt.xlim(0, max_val)
        plt.ylim(0, max_val)
        plt.title(f"{self.name}  Error=${avg_error:,.2f}  RMSLE={rmsle:.2f}  Hits={hit_rate:.1f}%")
        plt.show()

    @classmethod
    def quick_test(cls, predictor: Callable, data: List[Dict[str, Any]]) -> None:
        """Convenience method to create a tester and run immediately."""
        tester = cls(predictor, data)
        tester.run_test()

In [ ]:
print(dir(PricePredictorTester))

In [ ]:
print(f"Dataset type: {type(test)}")
print(f"First item type: {type(test[0])}")
print(f"First item keys: {test[0].keys()}")
print(f"Sample prompt: {test[0]['prompt'][:100]}...")
print(f"Sample completion: {test[0]['completion']}")

In [ ]:
test_list = list(test)   # ensure it's a list of dicts
PricePredictorTester.quick_test(weighted_topk_price_prediction, test_list)